# Testing the redis client components
start the server with: `uv run --with jupyter jupyter lab`
and run a redis container: `docker run --name redis-test -p 6379:6379 -d redis:7`

In [ ]:
import os

os.chdir("../..")
from src.kelder_api.components.redis_client.redis_client import RedisClient

In [2]:
redis_client = RedisClient()

async with redis_client.get_connection() as redis:
    ping = await redis.ping()
    print(ping)

True


In [3]:
await redis_client.write_value("hello", "2")
result = await redis_client.read_value("hello")
print(result)

2


In [5]:
from pydantic import BaseModel, Field, PrivateAttr
from datetime import datetime


class GPS(BaseModel):
    timestamp: datetime = Field(default_factory=datetime.now)
    coordinate: str


gps = GPS(coordinate="hello")

await redis_client.write_set("gps", gps)

In [13]:
data = await redis_client.read_set("gps")
gps_measurements = [GPS(**measurement) for measurement in data]
print(gps_measurements)
print(len((gps_measurements)))

+inf
[1755938148.081258, 1755096304.055606, 1755096086.595142, 1755032561.14055, 1755032540.044437, 1755032065.947365, 1755031998.504078, 1755031982.732478, 1755031923.34013, 1755031633.729909, 1755031570.699152, 1755031411.137944]
[GPS(timestamp=datetime.datetime(2025, 8, 23, 9, 35, 48, 81258), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 13, 15, 45, 4, 55606), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 13, 15, 41, 26, 595142), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 22, 2, 41, 140550), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 22, 2, 20, 44437), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 21, 54, 25, 947365), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 21, 53, 18, 504078), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 21, 53, 2, 732478), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 21, 52, 3, 340130), coordinate='he

In [15]:
date_start = datetime(2025, 8, 12, 22, 2, 20, 44437)
date_end = datetime.now()

data_ranged = await redis_client.read_set("gps", [date_start, date_end])

gps_measurements = [GPS(**measurement) for measurement in data_ranged]
print(gps_measurements)
print(len(gps_measurements))

1755938659.522616
[1755938148.081258, 1755096304.055606, 1755096086.595142, 1755032561.14055, 1755032540.044437]
[GPS(timestamp=datetime.datetime(2025, 8, 23, 9, 35, 48, 81258), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 13, 15, 45, 4, 55606), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 13, 15, 41, 26, 595142), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 22, 2, 41, 140550), coordinate='hello'), GPS(timestamp=datetime.datetime(2025, 8, 12, 22, 2, 20, 44437), coordinate='hello')]
5
